# 023 — Planificación clásica con STRIPS y PDDL

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("workflow", seed=23)
assert result["kind"] == "workflow"
assert result["evidence"]
show(result)


## Solución 1 — Operador desapilar

```text
desapilar(b, x):
  PRE: {sobre(b,x), libre(b)}
  ADD: {sobreMesa(b), libre(x)}
  DEL: {sobre(b,x)}
```

Justificación de DEL: `sobre(b,x)` deja de ser cierto (el bloque ya no está
ahí). NO se borra `libre(b)` (sigue libre en la mesa) y no hay que borrar
nada más: la suposición STRIPS conserva el resto. Un DEL de más rompe tanto
como un DEL de menos: borrar `libre(b)` dejaría a `b` inutilizable para
siempre.


## Solución 2 — RESULT literal a literal

a) `PRE = {sobre(C,A), libre(C)} ⊆ s` ✔ — aplicable.

b) `s \ DEL` quita `sobre(C,A)`; `∪ ADD` añade `sobreMesa(C)` y `libre(A)`:

```text
s' = {sobreMesa(A), sobreMesa(B), sobreMesa(C), libre(A), libre(B), libre(C)}
```

c) `apilar(B, C)` con PRE `{sobreMesa(B), libre(B), libre(C)}` ⊆ s' ✔ —
aplicable. `apilar(B, B)` la bloquea una **restricción de desigualdad**
(`?b ≠ ?destino`): en PDDL se expresa con `(not (= ?b ?x))` o tipando los
parámetros; si el modelador la omite, el planificador PUEDE generar el plan
absurdo — el error está en el modelo, no en el algoritmo.


In [ ]:
s = {"sobre(C,A)", "sobreMesa(A)", "sobreMesa(B)", "libre(C)", "libre(B)"}
PRE = {"sobre(C,A)", "libre(C)"}
ADD = {"sobreMesa(C)", "libre(A)"}
DEL = {"sobre(C,A)"}
assert PRE <= s
resultado = (s - DEL) | ADD
assert resultado == {"sobreMesa(A)", "sobreMesa(B)", "sobreMesa(C)",
                     "libre(A)", "libre(B)", "libre(C)"}
print("RESULT verificado ✔")


## Solución 3 — Dominio del workflow

```text
(:action validar
  :precondition (status_received)
  :effect (and (status_validated) (not (status_received))))
(:action esperar_aprobacion
  :precondition (status_validated)
  :effect (and (status_waiting_approval) (not (status_validated))))
(:action completar
  :precondition (status_waiting_approval)
  :effect (and (status_completed) (approved) (not (status_waiting_approval))))

(:init (status_received))   (:goal (status_completed))
```

El plan único es `validar; esperar_aprobacion; completar`, y `events` del
laboratorio reproduce exactamente esas tres transiciones.


In [ ]:
result = run_lab("workflow", seed=23)
assert result["result"]["events"] == [
    {"from": "received", "to": "validated"},
    {"from": "validated", "to": "waiting_approval"},
    {"from": "waiting_approval", "to": "completed"},
]
assert result["result"]["approved"] is True
print("plan y traza coinciden ✔")


## Solución 4 — Anomalía de Sussman (versión suave)

a) Lograr `sobre(A,B)` primero exige despejar A (desapilar C) y apilar A en B:

```text
desapilar(C,A); apilar(A,B)   → sobre(A,B) ✔ ... pero para sobre(B,C)
hay que mover B, y B está DEBAJO de A → desapilar(A,B) (deshace la meta),
apilar(B,C), apilar(A,B): 5 acciones.
```

b) Óptimo (3 acciones): `desapilar(C,A); apilar(B,C); apilar(A,B)` — construir
la torre **de abajo hacia arriba** (submeta `sobre(B,C)` antes que `sobre(A,B)`).

c) Las metas conjuntivas **interactúan**: satisfacerlas una a una en orden
arbitrario puede obligar a destruir lo logrado. Por eso la planificación no se
reduce a concatenar búsquedas por submeta, y por eso las heurísticas que
ignoran interacciones (delete relaxation) subestiman el costo real.


## Reflexión

1. El workflow del laboratorio tiene un único plan válido. ¿Qué propiedad de sus acciones (mira las listas PRE/DEL) elimina toda ramificación, y qué cambiaría si dos acciones fueran aplicables en el mismo estado?
2. La suposición STRIPS dice que lo no mencionado persiste. ¿Qué error concreto aparecería en el mundo de bloques si olvidas poner `libre(y)` en la lista DEL de `mover(b, x, y)`?
3. ¿Por qué 'el plan salió bien en el modelo' no garantiza nada sobre el mundo real, y qué componente (ausente en el laboratorio) convierte un planificador en un sistema utilizable?
